## 15.0.0
### Breaking

* drop support for python 3.10 and add support for python 3.13
    * library `ciso8601` ist rausgeflogen, weil python ab 3.11 das genauso schnell kann (see: https://github.com/fkie-cad/Logprep/blob/main/logprep/util/time.py#L36)
    * python 3.11 Sprachfeatures können genutzt werden (see: https://docs.python.org/3/whatsnew/3.11.html)
    * gute Chance, dass python 3.13 im `http Konnektor`schneller ist (möglicherweise bis zu 30%) -> liegt an `msgspec` library


* `CriticalInputError` is raised when the input preprocessor values can't be set, this was so far only true
  for the hmac preprocessor, but is now also applied for all other preprocessors.

* fix `delimiter` typo in `StringSplitterRule` configuration

old: 

```yaml
    filter: message
    string_splitter:
        source_fields: ["message"]
        target_field: result
        delimeter: ","
    description: '...'
```

new:

```yaml
    filter: message
    string_splitter:
        source_fields: ["message"]
        target_field: result
        delimiter: ","
    description: '...'
```

* removed the configuration `tld_lists` in `domain_resolver`, `domain_label_extractor` and `pseudonymizer` as
the list is now fixed inside the packaged logprep
* remove SQL feature from `generic_adder`, fields can only be added from rule config or from file
* use a single rule tree instead of a generic and a specific rule tree
    * hat sich rausgestellt, dass das keiner benutzt und die Benutzung keinen Vorteil mehr bringt

old:

```yaml
- fieldmanagername:
    type: field_manager
    specific_rules:
        - tests/testdata/specific/rules
    generic_rules:
        - tests/testdata/generic/rules
```

new:

```yaml
- fieldmanagername:
    type: field_manager
    rules:
        - tests/testdata/rules/rules
```


* replace the `extend_target_list` parameter with `merge_with_target` for improved naming clarity
and functionality across `FieldManager` based processors (e.g., `FieldManager`, `Clusterer`,
`GenericAdder`).
    * machte Sinn, weil wir der Funktion das Mergen von Dicts hinzugefügt haben

### Features

* configuration of `initContainers` in logprep helm chart is now possible

### Improvements

* fix `requester` documentation
* replace `BaseException` with `Exception` for custom errors
* refactor `generic_resolver` to validate rules on startup instead of application of each rule
* regex pattern lists for the `generic_resolver` are pre-compiled
* regex matching from lists in the `generic_resolver` is cached
* matching in the `generic_resolver` can be case-insensitive
* rewrite the helper method `add_field_to` such that it always raises an `FieldExistsWarning` instead of return a bool.
* add new helper method `add_fields_to` to directly add multiple fields to one event
* refactored some processors to make use of the new helper methods
* add `pre-commit` hooks to the repository, install new dev dependency and run `pre-commit install` in the root dir
* the default `securityContext`for the pod is now configurable
* allow `TimeParser` to get the current time with a specified timezone instead of always using local time and setting the timezone to UTC
* remove `tldextract` dependency
* remove `urlextract` dependency
* fix wrong documentation for `timestamp_differ`
* `FieldManager` supports merging dictionaries

* add sbom to images build in ci pipeline
    * was ist das? Warum brauchen wir das?

* add container signatures to images build in ci pipeline
    * wie funktioniert das eigentlich und warum ist das Wichtig?

### Bugfix

* fix `confluent_kafka.store_offsets` if `last_valid_record` is `None`, can happen if a rebalancing happens
  before the first message was pulled.
* fix pseudonymizer cache metrics not updated
* fix incorrect timezones for log arrival time and delta time in input preprocessing
* fix `_get_value` in `FilterExpression` so that keys don't match on values
* fix `auto_rule_tester` to work with `LOGPREP_BYPASS_RULE_TREE` enabled
* fix `opensearch_output` not draining `message_backlog` on shutdown
* silence `FieldExists` warning in metrics when `LOGPREP_APPEND_MEASUREMENT_TO_EVENT` is active